---
toc: true
image: example.png
pub-info:
    abstract: |
        Borrows an idea from process mining - the directly-follows graph - to turn a vidigi event log
        into a process-map style diagram of your model. It's a useful complement to the animation itself
        for communicating a process to non-technical stakeholders, and for verifying and debugging a
        model's logic.
execute: 
  enabled: true
---

In [ ]:
#| echo: false

import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook"

# Process-Map Style Outputs (Directly Follows Graphs - DFGs)

{{< include ../vidigi_2_0_0_example_warning.md >}}

Packages like bupaR (for R) and pm4py (for Python) are designed for process mining on real-world process data. 

However, there are a lot of useful visualisations that we can borrow from the process mining world and adapt for simulation, aiding in communicating processes with non-technical stakeholders, as well as verifying and debugging our model at various levels of granularity. 

What do we mean by a process map? Well, it could be an output like this:

![](process_map_image.png)

This gives us a quick overview of 

- the number of entities that reach each stage (the numbers in the 'nodes' - the boxes)
- the number of people moving between each stage (the numbers on the 'edges' - the links)
- all of the possible pathways observed through the system
- the relative likeliness of each branch at each point (the thickness of the lines between the 'edges')

Vidigi allows for the creation of these types of outputs in three different ways: 
- static outputs using the Graphviz library
- interactive outputs in jupyter notebooks using ipycytoscape
- interactive outputs in streamlit web apps using streamlit-cytoscape

In this section, we'll explore each of these. 

We'll start with a set of standard imports. 

In [ ]:
import pandas as pd

## Example 1 - A simple linear pathway

In [ ]:
from logger_model import Trial, g

In [ ]:
#| echo: false
#| output: asis
# Path to the external Python script
import os

file_path = "logger_model.py"

# Read the file content
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        code_content = f.read()
else:
    code_content = "File not found."
with open(file_path, "r") as f:
    code_content = f.read()

# Print the Quarto `{details}` block for collapsible output
print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code, which has had logging steps added at the appropriate points in the 'model' class

```python
{code_content}
```

:::

""")

In [ ]:
my_trial = Trial()

my_trial.run_trial()

Because of how this model has been set up, we have an attribute containing a list of EventLogger objects. We'll pull back the first one, representing the first run. 

In [ ]:
single_run = my_trial.all_event_logs[0]

single_run

In [ ]:
single_run.generate_dfg()

`generate_dfg` also takes a `warm_up=` parameter, for discarding an initial period before the graph is built - useful for the same reason it's useful anywhere else in vidigi, but with a caveat specific to process maps: this is a plain time-based filter on individual rows, not the presence-aware trimming `reshape_for_animations`/`animate_activity_log` use (see [feat_animation_warm_up.ipynb](../feat_animation_warm_up/feat_animation_warm_up.ipynb) for that one). A case that falls entirely inside the warm-up is dropped completely, and a case that spans the cutoff loses the single edge connecting its last pre-cutoff event to its first post-cutoff one, since one side of that pair is no longer in the log - both deliberate, since `discover_dfg` builds each case's edges from its own consecutive rows, so dropping early rows can't make a case silently vanish from output it should still appear in the way the animation functions' own naive-filtering trap can. Below, the same run's `arrival` count drops from 132 to 113 once the first 100 time units are excluded - the 19 patients who arrived, and in most cases departed, before the cutoff:

In [ ]:
single_run.generate_dfg(warm_up=100)

We can generate a different type of graph, passing in any arguments accepted by the relevant function. 

The accepted arguments can be found in the following places:

- Graphviz outputs: [here](../../reference/process_mapping.dfg_to_graphviz.qmd)
- Jupyter cytoscape outputs: [here](../../reference/process_mapping.dfg_to_cytoscape.qmd)
- Streamlit cytoscape outputs: [here](../../reference/process_mapping.dfg_to_cytoscape_streamlit.qmd)

In [ ]:
single_run.generate_dfg(output_format="cytoscape-jupyter", spacing_factor=2)

### Sim-focussed node metrics: queue build-up and resource load

The node counts so far are process mining's native currency - how many entities *passed through* each step. A simulation can tell us something process mining usually can't: how many entities were *sitting at* each step at any given moment. Passing `occupancy_metrics=True` adds, to every queue and resource node, the mean, minimum and maximum number of entities present - queue length for `queue` steps, units in use for `resource_use` steps.

It's off by default because the queue figures are reconstructed the same way `animate_activity_log` rebuilds each frame (via `reshape_for_animations`), which is slow on a long log. `occupancy_snapshot_interval=` trades resolution for speed, and any `warm_up=` is applied to these figures too.

In [ ]:
single_run.generate_dfg(occupancy_metrics=True)

The same figures are available as a plain table via `vidigi.analysis.activity_occupancy_stats` - for a report, or for merging onto a hand-built DFG with `discover_dfg(occupancy_stats=...)`. `across_runs=` chooses how a multi-run log is combined: `"average"` (the figure expected per replication) or `"pool"` (pooled across every run and snapshot, so the maximum is the worst seen in any run).

In [ ]:
from vidigi.analysis import activity_occupancy_stats

activity_occupancy_stats(single_run.to_dataframe())

### From a `TrialLogger`

Everything above was one run. A `TrialLogger` - vidigi's container for a whole trial - has its own `generate_dfg()` with three ways to handle the several replications it holds:

- **the default** is the *representative run*: the replication whose mean time in system is closest to the trial median
- `run_number=N` renders one named replication
- `across_runs=True` builds **one** combined cross-run map - transitions grouped per `(run, entity)` so no edge is fabricated across the seam between runs, node and edge counts shown *per replication* with the between-run range (`n=3.5 (1–7)`), and transition times pooled over every run

In [ ]:
from vidigi.logging import TrialLogger

trial_logs = TrialLogger(my_trial.all_event_logs)

# the representative run - closest to the trial's median time in system
trial_logs.generate_dfg(warm_up=100)

A single named replication:

In [ ]:
trial_logs.generate_dfg(run_number=7, warm_up=100)

`across_runs=True` combines every replication into a single map. The counts become per-run means (with the range across runs), and the title carries a reminder that this is aggregated simulation output rather than one observed run. Passing `warm_up=` matters here - without it the start-up transient is baked into a stakeholder-facing figure, so vidigi warns.

In [ ]:
trial_logs.generate_dfg(across_runs=True, warm_up=100)

:::{.callout-note}
The cross-run map is a summary, not a run of your model. A few things to keep in mind:

- entities still in the system when a run ends have **truncated paths**, which biases transition times downwards
- the cytoscape renderers hide any edge seen less than `min_frequency` times per run on average (default 1)
- `get_event_duration_ci` is the route to a formal confidence interval on a transition time, rather than the plain min–max range shown on the graph
:::

## Example 2 - A simple linear pathway - Without EventLogger, or for fine control over the DFG functions

First, we will need some extra imports. 

In [ ]:
from vidigi.process_mapping import (
    add_sim_timestamp,
    dfg_to_cytoscape,
    dfg_to_graphviz,
    discover_dfg,
)

First, let's read in an event log. This is a standard vidigi event log created with the EventLogger class, with the dataframe saved as a csv for easy portability.

In [ ]:
event_log = pd.read_csv("sample_event_log_10_day_10_run.csv")

**IMPORTANT** - the individual DFG functions (`discover_dfg` and friends) build transitions per entity without regard to which run a row belongs to. Handed an unfiltered multi-run log they fabricate edges across the seam between runs - because entity IDs are reused, an entity that departs in one run looks like it jumps straight back to an early stage in the next - and vidigi now warns when this happens.

For an `EventLogger` or `TrialLogger`, `generate_dfg()` (above) handles this for you. Using the functions directly, either filter to one run first (below), or pass `run_col_name="run"` to `discover_dfg` so it groups transitions by run.

In [ ]:
event_log_single = event_log[event_log["run"] == 1]

event_log_single.head()

:::{.callout-note}
Note that we could also do this for any situation where we want fine-grained control over the use of the individual DFG-creation functions, even if we have used EventLogger. 

In that case, we'd just get the event log in a pandas dataframe format using `event_log = my_event_logger_object.to_dataframe()`

The EventLogger object will already only contain a single run. If you are using TrialLogger, `trial_logs.generate_dfg(...)` (shown earlier) is usually easier; to drive the functions directly, filter to a single run or pass `run_col_name=` as mentioned above. 
:::

Before we can pass this to our DFG functions, we need to add a timestamp column. 

This is because the functions are designed to work regardless of the unit of time you've used for your simulation - but it's a lot easier if we encode that at the beginning of the process, and allows the dfg to calculate various summary statistics.

The defaults for column names reflect vidigi's EventLogger default, so if we've used EventLogger, we won't need to specify column names here. 

If we have a simulation where we don't really have a date in mind - for example, a simulation that shows no seasonality across the year - then we can leave the `sim_start` parameter blank, and it will choose a dummy date of 12am on the 1st January, 2000. This won't be displayed anywhere in the later visualisations, so you don't have to worry too much about it! 

In [ ]:
df_timestamp = add_sim_timestamp(event_log_single, time_unit="minutes")
df_timestamp.head()

Now we can use the `discover_dfg` function to turn the event log into the format required by the various static and interactive visualisation functions. 

In [ ]:
nodes, edges = discover_dfg(df_timestamp)

Let's explore what our nodes look like. 

In [ ]:
nodes

The edges dataframe then calculates the frequency of various transitions between nodes, the probability of each - here 1.0 as there are no branching steps - and various time-based metrics for each transition.

In [ ]:
edges

Finally, we can display the output. 

In [ ]:
dfg_to_graphviz(
    nodes,
    edges,
)

By default, it will display the mean time between steps. We could instead ask for the median: 

In [ ]:
dfg_to_graphviz(nodes, edges, time_metric="median")

In [ ]:
dfg_to_graphviz(nodes, edges, time_metric="standard_deviation")

In [ ]:
dfg_to_graphviz(nodes, edges, time_metric="max")

In [ ]:
dfg_to_graphviz(nodes, edges, time_metric="min")

# A more complex example - stroke dataset - without EventLogger

Again, make sure we've imported the relevant functions. 

In [ ]:
from vidigi.process_mapping import add_sim_timestamp, dfg_to_graphviz, discover_dfg

Let's do the same for a more complex example - a stroke ward with multiple branching pathways and patient types.

In [ ]:
stroke_df = pd.read_csv("event_log_stroke.csv")
stroke_df.head()

:::{.callout-note}
Again, note that we could also do this for any situations where we want fine-grained control over the use of the individual DFG-creation functions, event if we have used EventLogger. 

In that case, we'd just get the event log in a pandas dataframe format using `event_log = my_event_logger_object.to_dataframe()`

The EventLogger object will already only contain a single run. If you are using TrialLogger, make sure you filter to a single run, as mentioned above. 
:::

Let's tidy it up a bit!

The event names use underscores, which won't wrap onto new lines, so we'll replace those with spaces. We can also remove _time everywhere it appears, as in this case it's not adding anything useful. 

In [ ]:
stroke_df["event"] = stroke_df["event"].apply(
    lambda x: x.replace("_time", "").replace("_", " ")
)

Again, we'll add our timestamp, then get our nodes and edges. 

In [ ]:
stroke_df_timestamp = add_sim_timestamp(stroke_df)

nodes, edges = discover_dfg(stroke_df_timestamp, case_col="id")

nodes

In [ ]:
edges

Let's see what this looks like. 

We'll also make use of a couple of extra parameters, filtering out any very infrequent paths. 

In [ ]:
g = dfg_to_graphviz(nodes, edges, min_frequency=5)

g

Let's take a quick look at what our output actually is. 

In [ ]:
type(g)

It's a graphviz Digraph object, which we could continue to modify if we wanted to. 

In [ ]:
with g.subgraph(name="cluster_0") as c:
    c.attr(color="blue")
    c.node_attr["style"] = "filled"
    # Note that we will need to include the newline indicator when filtering; we can work out where this is by visually observing
    # the node labels in the orignal diagram
    c.edges(
        [
            ("clock start", "nurse q start"),
            ("nurse q start", "nurse triage\nstart"),
            ("nurse triage\nstart", "nurse triage\nend"),
        ]
    )
    c.attr(label="Arrival and Triage")

g

We can also output to an image type - but note it will output as a series of bytes that will need to be converted into an image in some other way depending on your intended use.

In [ ]:
g_png = dfg_to_graphviz(nodes, edges, min_frequency=5, return_image=True, format="png")

type(g_png)

In [ ]:
from IPython.display import Image

Image(g_png)

We can also change the orientation of the output. 

In [ ]:
dfg_to_graphviz(nodes, edges, min_frequency=5, direction="TD")

While the early parts of the process are easy to interpret in minutes, it's hard to interpret the ward stay durations, which are much longer. 

We might actually want to split these into two separate graphs by filtering the node and edges table. 

For now, though, we're going to rediscover our dfg using a unit of days, recalculate our timestamp and node/edge dataframes, and visualise this again. Note where 'minutes' has changed to 'days'. 

In [ ]:
nodes, edges = discover_dfg(stroke_df_timestamp, case_col="id", time_unit="days")

dfg_to_graphviz(
    nodes,
    edges,
    min_frequency=5,
    time_unit="days",
)

### Looping through graphviz outputs

While a lot of value can be obtained from just checking that your model is behaving as expected, you can get even more out of these outputs by faceting by different elements. 

For example, here we expect patients with different stroke types to have different lengths of stay. 

We need to import the `display` function from `IPython.display` and explicitly call this to ensure it's included. 

In [ ]:
from IPython.display import display

for i in stroke_df_timestamp["patient_diagnosis_type"].unique():
    df = stroke_df_timestamp[stroke_df_timestamp["patient_diagnosis_type"] == i]

    nodes, edges = discover_dfg(df, case_col="id", time_unit="days")

    display(dfg_to_graphviz(nodes, edges, time_unit="days", title=f"Stroke Type: {i}"))

We could do what we talked about earlier and try filtering the nodes and edges. 

In [ ]:
from IPython.display import display

for i in stroke_df_timestamp["patient_diagnosis_type"].unique():
    df = stroke_df_timestamp[stroke_df_timestamp["patient_diagnosis_type"] == i]

    nodes, edges = discover_dfg(df, case_col="id", time_unit="days")

    # Remember that we will need to include the newline indicator when filtering; we can work out where this is by visually observing
    # the node labels in the orignal diagram
    steps_to_include = [
        "ct or ctp scan\nend",
        "sdec admit",
        "sdec discharge",
        "ward q start",
        "ward admit",
        "ward discharge",
        "exit",
    ]

    nodes = nodes[nodes["activity"].isin(steps_to_include)]

    edges = edges[
        (edges["source"].isin(steps_to_include))
        | (edges["target"].isin(steps_to_include))
    ]

    display(dfg_to_graphviz(nodes, edges, time_unit="days", title=f"Stroke Type: {i}"))

We could check out the maximum durations too. 

In [ ]:
for i in stroke_df_timestamp["patient_diagnosis_type"].unique():
    df = stroke_df_timestamp[stroke_df_timestamp["patient_diagnosis_type"] == i]

    nodes, edges = discover_dfg(df, case_col="id", time_unit="days")

    display(
        dfg_to_graphviz(
            nodes, edges, time_unit="days", title=f"Stroke Type: {i}", time_metric="max"
        )
    )

### Interactive outputs

Finally, let's take a look at an interactive output. 

Here, we pass in the same nodes and edges. However, we need to choose a layout; 'dagre' is often the best option for getting something that looks similar to graphviz. 

In [ ]:
nodes, edges = discover_dfg(
    stroke_df_timestamp, case_col="id", timestamp_col="timestamp", time_unit="days"
)

op = dfg_to_cytoscape(
    nodes,
    edges,
    layout_name="dagre",
    layout_orientation="LR",
    spacing_factor=1.5,
    width=1400,
    time_unit="days",
)


display(op)

You can also try this out in the following Streamlit app. 

```{=html}
<iframe width='780' height='500' src='https://vidigi-dfg-demo.streamlit.app/?embed=true' title='Demo app'></iframe>
```